In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/telco_clean.csv")

In [3]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]

# 1. How many add-on services the customer has
df["num_addons"] = (df[addons] == "Yes").sum(axis=1)

# 2. Has security or tech support (both were strongly protective in the EDA)
df["has_security_support"] = ((df["OnlineSecurity"] == "Yes") |
                              (df["TechSupport"] == "Yes")).astype(int)

# 3. Pays automatically (bank transfer / credit card) vs manually
df["auto_pay"] = df["PaymentMethod"].str.contains("automatic").astype(int)

# 4. Tenure band (helps linear models capture the non-linear tenure effect)
df["tenure_group"] = pd.cut(df["tenure"], [-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"]).astype(str)

# Do the new features carry signal? (churn rate per value)

In [4]:
for col in ["num_addons", "has_security_support", "auto_pay", "tenure_group"]:
    print(df.groupby(col)["Churn"].mean().round(3), "\n")

num_addons
0    0.214
1    0.458
2    0.358
3    0.274
4    0.223
5    0.124
6    0.053
Name: Churn, dtype: float64 

has_security_support
0    0.334
1    0.171
Name: Churn, dtype: float64 

auto_pay
0    0.347
1    0.160
Name: Churn, dtype: float64 

tenure_group
0-12     0.474
13-24    0.287
25-48    0.204
49-72    0.095
Name: Churn, dtype: float64 



# Define feature groups for the Step 06 pipeline

In [5]:
target = "Churn"
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "num_addons"]
binary_features = ["SeniorCitizen", "has_security_support", "auto_pay"]
categorical_features = [c for c in df.columns
                        if c not in numeric_features + binary_features + [target]]
print("categorical:", categorical_features)

df.to_csv("../data/processed/telco_features.csv", index=False)
print(df.shape)

categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']
(7043, 24)
